# Q2. Unsupervised Learning — Customer Segmentation

**Goal:** Discover natural customer segments using K-Means clustering, then visualise them using PCA.  
**Dataset:** `q2_customers.csv` — 500 customers, 6 behavioural features, no target label.

## Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
print('Libraries loaded successfully.')

---
## Task 1 — Data Preparation

We load the data and scale all features using `StandardScaler` before applying K-Means.

In [ ]:
df = pd.read_csv('../data/q2_customers.csv')

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print('\nMissing values:')
print(df.isnull().sum())
print('\nSummary statistics:')
df.describe().round(2)

### Why Scaling is Essential Before K-Means

K-Means assigns each point to the nearest cluster centre using **Euclidean distance**. If features are on very different scales, the algorithm is dominated by whichever feature has the largest numerical range — not because it's more important, but simply because its distances are larger.

In this dataset, `annual_spend` ranges from ~5,000 to ~120,000, while `visits_per_month` ranges from 1 to ~20. Without scaling, annual spend would completely overwhelm the distance calculations, and features like `visits_per_month` or `num_categories_purchased` would contribute almost nothing to cluster formation.

`StandardScaler` transforms each feature to have **mean = 0 and standard deviation = 1**, putting all features on equal footing so every dimension contributes fairly to the clustering.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

print(f'Scaled data shape: {X_scaled.shape}')
print(f'Mean of scaled features (should be ~0): {X_scaled.mean(axis=0).round(4)}')
print(f'Std  of scaled features (should be ~1): {X_scaled.std(axis=0).round(4)}')

---
## Task 2 — Choosing K: The Elbow Method

We compute the **Within-Cluster Sum of Squares (WCSS)** for K = 1 to 10. WCSS measures how tightly packed each cluster is — lower is better, but it always decreases as K increases. The 'elbow' is the point where adding another cluster gives diminishing returns.

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

# Plot the elbow curve
plt.figure(figsize=(8, 4))
plt.plot(K_range, wcss, marker='o', linewidth=2, color='steelblue', markersize=7)
plt.axvline(x=4, color='tomato', linestyle='--', linewidth=1.5, label='Optimal K = 4')
plt.title('Elbow Method — WCSS vs Number of Clusters', fontsize=13, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.xticks(K_range)
plt.legend()
plt.tight_layout()
plt.show()

# Print WCSS values and the drop at each step
print('K   | WCSS      | Drop from previous')
print('-'*42)
for i, (k, w) in enumerate(zip(K_range, wcss)):
    drop = f'{wcss[i-1]-w:.1f}' if i > 0 else 'N/A'
    print(f'K={k}  | {w:>8.1f}  | {drop}')

### Optimal K Selection

The WCSS drops sharply from K=1 (3000.0) to K=2 (969.0) and again to K=3 (561.3). The most significant **elbow** occurs at **K=4** — the drop from K=3 to K=4 is 116.3, but from K=4 to K=5 it falls to just 42.6. After K=4, each additional cluster yields diminishing improvements in compactness.

**Chosen K = 4.** This balances cluster quality with interpretability — four segments is also a practical number for a marketing team to act on.

---
## Task 3 — K-Means Clustering (K=4)

We fit K-Means with K=4, assign cluster labels, and inspect the cluster centroids to understand what each group represents in business terms.

In [ ]:
km = KMeans(n_clusters=4, random_state=42, n_init=10)
km.fit(X_scaled)

# Add cluster labels to the original dataframe
df['cluster'] = km.labels_

print('Cluster sizes:')
print(df['cluster'].value_counts().sort_index())

# Decode centroids back to original scale for interpretability
centroids_orig = scaler.inverse_transform(km.cluster_centers_)
centroids_df = pd.DataFrame(
    centroids_orig,
    columns=[c for c in df.columns if c != 'cluster']
).round(1)
centroids_df.index.name = 'Cluster'

print('\nCluster Centroids (original scale):')
centroids_df

### Cluster Interpretation (Business Terms)

| Cluster | Age | Annual Spend | Visits/Month | Basket Size | Days Since Last Visit | Categories | Business Label |
|---|---|---|---|---|---|---|---|
| 0 | 25 | £14,847 | 14.3 | £559 | 9 days | 2.1 | 🟢 Young Frequent Browsers |
| 1 | 57 | £89,814 | 2.5 | £5,296 | 148 days | 7.5 | 🔴 Lapsed High-Value Customers |
| 2 | 40 | £43,341 | 8.2 | £2,022 | 35 days | 4.4 | 🔵 Mid-Tier Regular Shoppers |
| 3 | 57 | £89,036 | 2.6 | £5,751 | 65 days | 7.5 | 🟡 Active High-Value Customers |

**Cluster 0 — Young Frequent Browsers:** Young customers (~25) who visit very often (14×/month) but spend little per visit (basket ~£559) and buy from only 2 categories. High engagement but low monetisation — potential to upsell with targeted promotions.

**Cluster 1 — Lapsed High-Value Customers:** Older customers (~57) with high historical spend (~£90k/year) and wide category breadth (7.5), but they haven't visited in ~148 days. These are at-risk churners who were once premium shoppers — win-back campaigns are critical here.

**Cluster 2 — Mid-Tier Regular Shoppers:** Middle-aged customers (~40) with moderate spend, regular visits (8×/month), and average basket sizes. The backbone of the customer base — loyalty rewards could move them toward the high-value segments.

**Cluster 3 — Active High-Value Customers:** Older customers (~57) who spend as much as Cluster 1 (~£89k) and buy across 7.5 categories, but crucially visited just 65 days ago — still engaged. These are the most valuable active customers and deserve VIP retention treatment.

---
## Task 4 — Dimensionality Reduction with PCA

PCA compresses the 6 original features into 2 principal components (PC1, PC2) that capture the maximum variance in the data. This lets us plot all 500 customers on a 2D chart without losing too much information.

In [ ]:
# Apply PCA to reduce to 2 components
# Note: PCA is applied to X_scaled (without the cluster column)
features = [c for c in df.columns if c != 'cluster']
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# Explained variance
print('--- Explained Variance Ratio ---')
for i, var in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {var:.4f}  ({var*100:.2f}% of variance)')
print(f'  Total: {pca.explained_variance_ratio_.sum():.4f}  ({pca.explained_variance_ratio_.sum()*100:.2f}%)')

# Feature loadings
print('\n--- Feature Loadings (Components) ---')
loadings_df = pd.DataFrame(
    pca.components_,
    columns=features,
    index=['PC1', 'PC2']
).round(4)
loadings_df

### PCA Interpretation

**Explained Variance:**  
PC1 captures **83.56%** of the total variance in the dataset — an exceptionally high proportion for a single component. PC2 adds another 5.57%, bringing the total to ~89%. This means projecting onto just 2 dimensions retains nearly 90% of all information, making the 2D scatter plot a reliable representation of the full 6-dimensional space.

**What PC1 captures:**  
All six features load positively and with similar magnitude on PC1 (loadings: age ≈ 0.41, annual_spend ≈ 0.42, visits ≈ -0.41, basket ≈ 0.41, days_since ≈ 0.38, categories ≈ 0.41). PC1 is essentially a **general customer value axis** — customers with high PC1 scores tend to be older, spend more, have larger baskets and broader category breadth, but visit less frequently.

**What PC2 captures:**  
`days_since_last_visit` dominates PC2 with a loading of 0.91, while all other features contribute minimally. PC2 is essentially a **recency axis** — it separates recently active customers (low PC2) from lapsed customers (high PC2). This explains why Clusters 1 and 3 (both high-value) are separated in the scatter plot: they differ mainly on recency.

---
## Task 5 — Cluster Visualisation (PCA Scatter Plot)

We plot each customer as a point in the 2D PCA space, coloured by their assigned cluster label.

In [ ]:
cluster_labels = {
    0: 'Cluster 0: Young Frequent Browsers',
    1: 'Cluster 1: Lapsed High-Value',
    2: 'Cluster 2: Mid-Tier Regular',
    3: 'Cluster 3: Active High-Value'
}
colours = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']

fig, ax = plt.subplots(figsize=(9, 6))

for cluster_id, colour in zip(sorted(df['cluster'].unique()), colours):
    mask = df['cluster'] == cluster_id
    ax.scatter(
        X_pca[mask, 0],
        X_pca[mask, 1],
        c=colour,
        label=cluster_labels[cluster_id],
        alpha=0.7,
        edgecolors='white',
        linewidth=0.4,
        s=60
    )

# Plot cluster centroids in PCA space
centroids_pca = pca.transform(km.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
           c='black', marker='X', s=180, zorder=5, label='Centroids')

ax.set_title('Customer Segments — K-Means Clusters in PCA Space',
             fontsize=13, fontweight='bold')
ax.set_xlabel('PC1 — General Customer Value Axis (83.6% variance)', fontsize=11)
ax.set_ylabel('PC2 — Recency Axis (5.6% variance)', fontsize=11)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()

print('Cluster visualisation complete.')

### Visualisation Interpretation

The scatter plot clearly shows **four well-separated customer groups** in PCA space:

- **Cluster 0 (blue)** sits in the bottom-left — low PC1 (low overall value) and low PC2 (recent visitors). These are the young, frequent, low-spending customers.
- **Cluster 2 (green)** occupies the centre — moderate PC1 scores, the mid-tier regulars.
- **Cluster 3 (orange)** is in the top-right with high PC1 but moderate PC2 — high-value customers who are still recently active.
- **Cluster 1 (red)** is in the far top-right with very high PC2 — these are the lapsed high-value customers, separated from Cluster 3 purely by recency.

The fact that clusters are visually well-separated confirms that K=4 was a good choice — the groups are genuinely distinct in the feature space, not just artefacts of the algorithm.